# Litter Detection — Analysis Notebook

1. Inspect training data (images + masks)
2. Review MLflow experiment history
3. Run inference with the best trained model

**Prerequisites**: `uv run python prepare.py` and at least one `uv run python train.py` run.

In [ ]:
import json, random, sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image
import albumentations as A
from albumentations.pytorch import ToTensorV2

sys.path.insert(0, '.')

DATA_DIR   = Path('data')
IMAGES_DIR = DATA_DIR / 'images'
MASKS_DIR  = DATA_DIR / 'masks'
MODEL_PATH = Path('best_model.pth')
INFER_SIZE = 384

print('data/ exists:        ', DATA_DIR.exists())
print('best_model.pth exists:', MODEL_PATH.exists())

: 

## 1. Dataset overview

In [ ]:
print(json.dumps(json.loads((DATA_DIR/'meta.json').read_text()), indent=2))
train_stems = (DATA_DIR/'train.txt').read_text().splitlines()
val_stems   = (DATA_DIR/'val.txt').read_text().splitlines()
print(f'\nTrain: {len(train_stems)}   Val: {len(val_stems)}')

## 2. Sample images + masks

In [ ]:
def show_samples(stems, n=8, title='Samples'):
    chosen = random.sample(stems, min(n, len(stems)))
    fig, axes = plt.subplots(2, len(chosen), figsize=(len(chosen)*2.5, 5))
    fig.suptitle(title, fontsize=13)
    for col, stem in enumerate(chosen):
        img  = Image.open(IMAGES_DIR/f'{stem}.jpg')
        mask = np.array(Image.open(MASKS_DIR/f'{stem}.png')) > 127
        axes[0,col].imshow(img); axes[0,col].axis('off')
        axes[0,col].set_title(stem, fontsize=7)
        overlay = np.array(img).copy().astype(float)
        overlay[mask] = (overlay[mask]*0.4 + np.array([255,80,0])*0.6).clip(0,255)
        axes[1,col].imshow(overlay.astype(np.uint8)); axes[1,col].axis('off')
        axes[1,col].set_title(f'{mask.mean():.1%} litter', fontsize=7)
    plt.tight_layout(); plt.show()

show_samples(train_stems, title='Training samples')
show_samples(val_stems,   title='Validation samples')

## 3. Mask statistics

In [ ]:
fracs = np.array([
    (np.array(Image.open(MASKS_DIR/f'{s}.png')) > 127).mean()
    for s in (train_stems + val_stems)
])
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(fracs, bins=50, edgecolor='black')
axes[0].set(xlabel='Litter pixel fraction', ylabel='Count',
            title='Distribution of litter coverage per image')
axes[1].hist(fracs, bins=50, cumulative=True, density=True, edgecolor='black')
axes[1].set(xlabel='Litter pixel fraction', ylabel='Cumulative density', title='CDF')
plt.tight_layout(); plt.show()
print(f'Mean: {fracs.mean():.3%}  Median: {np.median(fracs):.3%}  '
      f'Empty: {(fracs==0).sum()}  >50%: {(fracs>0.5).sum()}')

## 4. MLflow experiment history

In [ ]:
import mlflow, pandas as pd

client = mlflow.MlflowClient()
exp = client.get_experiment_by_name('litter-segmentation')
if exp is None:
    print('No experiment yet — run train.py first.')
else:
    runs = client.search_runs(
        experiment_ids=[exp.experiment_id],
        filter_string="attributes.status = 'FINISHED'",
        order_by=['metrics.best_val_iou DESC']
    )
    df = pd.DataFrame([{
        'run_name':     r.data.tags.get('mlflow.runName', r.info.run_id[:8]),
        'best_val_iou': r.data.metrics.get('best_val_iou', float('nan')),
        'elapsed_min':  round(r.data.metrics.get('elapsed_s', 0)/60, 1),
        'lr':           r.data.params.get('lr', '?'),
        'batch_size':   r.data.params.get('batch_size', '?'),
        'encoder':      r.data.params.get('encoder_channels', '?'),
    } for r in runs])
    display(df.style.highlight_max(subset=['best_val_iou'], color='lightgreen')
              .format({'best_val_iou': '{:.4f}'}))

In [ ]:
# Bar chart of best val_iou per run
if exp:
    run_bests = sorted(
        [(r.data.metrics.get('best_val_iou', 0),
          r.data.tags.get('mlflow.runName', r.info.run_id[:8]))
         for r in runs if r.data.metrics.get('best_val_iou') is not None]
    )
    vals  = [x[0] for x in run_bests]
    names = [x[1] for x in run_bests]
    best  = max(vals)
    colours = ['tomato' if v == best else 'steelblue' for v in vals]
    fig, ax = plt.subplots(figsize=(10, max(4, len(names)*0.32)))
    ax.barh(names, vals, color=colours)
    ax.axvline(0.65, color='green', linestyle='--', alpha=0.7, label='target 0.65')
    ax.set(xlabel='best_val_iou', title='Best val_iou per run (red = best)')
    ax.legend(); plt.tight_layout(); plt.show()

## 5. Load best model

In [ ]:
from train import *

def get_device():
    if torch.cuda.is_available():         return torch.device("cuda")
    if torch.backends.mps.is_available(): return torch.device("mps")
    return torch.device("cpu")

device = get_device()
model  = None

# ── Choose which model to load ─────────────────────────────────────────
# CHECKPOINT = "best_model.pth"           # overall best  (ResNet34, iou=0.6738)
# CHECKPOINT = "best_resnet34.pth"        # ResNet34       (iou=0.6738)
# CHECKPOINT = "best_efficientnetb4.pth" # EfficientNet-B4 (iou=0.6323)
CHECKPOINT  = "best_model.pth"
MODEL_CLASS = ResNet34UNet       # match to checkpoint above
# MODEL_CLASS = EfficientNetB4UNet

p = Path(CHECKPOINT)
if p.exists():
    model = MODEL_CLASS(dropout=DROPOUT).to(device)
    model.load_state_dict(torch.load(p, map_location=device))
    model.eval()
    n = sum(par.numel() for par in model.parameters())
    print(f"Loaded {CHECKPOINT} ({MODEL_CLASS.__name__}) on {device}  ({n:,} params)")
else:
    print(f"{CHECKPOINT} not found")


## 6. Inference helpers

In [ ]:
infer_tf = A.Compose([
    A.Resize(INFER_SIZE, INFER_SIZE),
    A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)),
    ToTensorV2(),
])

def timeit(method):
    def timed(*args, **kw):
        ts = time.time()
        result = method(*args, **kw)
        te = time.time()
        if 'log_time' in kw:
            name = kw.get('log_name', method.__name__.upper())
            kw['log_time'][name] = int((te - ts) * 1000)
        else:
            print('%r  %2.2f ms' % \
                  (method.__name__, (te - ts) * 1000))
        return result
    return timed

@timeit
def predict(src, threshold=0.5):
    """src = file path (str/Path) or PIL Image."""
    assert model is not None, 'Load a model first'
    img = np.array((Image.open(src) if isinstance(src,(str,Path)) else src).convert('RGB'))
    t = infer_tf(image=img)['image'].unsqueeze(0).to(device)
    with torch.no_grad():
        prob = torch.sigmoid(model(t)).squeeze().cpu().numpy()
    return img, prob, (prob > threshold).astype(np.uint8)

def show_prediction(src, threshold=0.5, gt_mask_path=None):
    img, prob, pred = predict(src, threshold)
    disp = np.array(Image.fromarray(img).resize((INFER_SIZE, INFER_SIZE)))
    ncols = 4 if gt_mask_path else 3
    fig, axes = plt.subplots(1, ncols, figsize=(ncols*4, 4))
    axes[0].imshow(disp);  axes[0].set_title('Input');          axes[0].axis('off')
    axes[1].imshow(prob, cmap='hot', vmin=0, vmax=1)
    axes[1].set_title('Probability'); axes[1].axis('off')
    ov = disp.copy().astype(float)
    ov[pred.astype(bool)] = (ov[pred.astype(bool)]*0.4 + np.array([255,80,0])*0.6).clip(0,255)
    axes[2].imshow(ov.astype(np.uint8)); axes[2].set_title('Prediction'); axes[2].axis('off')
    if gt_mask_path:
        gt = (np.array(Image.open(gt_mask_path).resize(
            (INFER_SIZE,INFER_SIZE), Image.NEAREST)) > 127).astype(np.uint8)
        axes[3].imshow(gt, cmap='gray'); axes[3].set_title('Ground truth'); axes[3].axis('off')
        inter = (pred*gt).sum(); union = ((pred+gt)>0).sum()
        fig.suptitle(f'IoU = {inter/max(union,1):.4f}', fontsize=12)
    plt.tight_layout(); plt.show()

print('predict() and show_prediction() ready')

## 7. Predictions on validation samples

In [ ]:
TEST_THRESHOLD = 0.7

if model is not None:
    for stem in random.sample(val_stems, min(6, len(val_stems))):
        show_prediction(IMAGES_DIR/f'{stem}.jpg',
                        threshold=TEST_THRESHOLD,
                        gt_mask_path=MASKS_DIR/f'{stem}.png')

## 8. Full validation IoU

In [ ]:


if model is not None:
    ious = []
    for stem in val_stems:
        _, _, pred = predict(IMAGES_DIR/f'{stem}.jpg', threshold=TEST_THRESHOLD)
        gt = (np.array(Image.open(MASKS_DIR/f'{stem}.png').resize(
            (INFER_SIZE,INFER_SIZE), Image.NEAREST)) > 127).astype(np.uint8)
        ious.append((pred*gt).sum() / max(((pred+gt)>0).sum(), 1))
    ious = np.array(ious)
    
    print(f'mean={ious.mean():.4f}  median={np.median(ious):.4f}  '
          f'min={ious.min():.4f}  max={ious.max():.4f}')
    
    plt.figure(figsize=(8,4))
    plt.hist(ious, bins=30, edgecolor='black')
    plt.axvline(ious.mean(), color='red', linestyle='--', label=f'mean={ious.mean():.3f}')
    plt.xlabel('IoU'); plt.ylabel('Count')
    plt.title('Per-image IoU on validation set')
    plt.legend(); plt.tight_layout(); plt.show()

## 9. Worst predictions (lowest IoU)

In [ ]:
if model is not None and 'ious' in dir():
    worst = sorted(zip(ious, val_stems))[:4]
    print('Worst 4 predictions:')
    for iou_val, stem in worst:
        print(f'  {stem}  iou={iou_val:.4f}')
        show_prediction(IMAGES_DIR/f'{stem}.jpg',
                        threshold=TEST_THRESHOLD,
                        gt_mask_path=MASKS_DIR/f'{stem}.png')

## 10. Inference on a custom image

Set `CUSTOM_IMAGE` to any file path and run.

In [ ]:
CUSTOM_IMAGE = './image3.jpeg'
show_prediction(CUSTOM_IMAGE, threshold=TEST_THRESHOLD)